# Import required packages. 

In [1]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

%matplotlib inline
from matplotlib import pyplot as plt

# Load the downloaed dataset CIFAR10 and normalize

In [2]:
data_path = '../torch_tutorial/data'
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5,0.5,0.5))
    ])
cifar10 = datasets.CIFAR10(data_path, train=True, download=False, transform=transform)
cifar10_test = datasets.CIFAR10(data_path, train=False, download=False,transform=transform )

Create sub datasets for only two classes: airplane or bird. label: 0 for airplaine, 1 for bird
The new datasets: cifar2 and cifar2_test. In a dataset, images are stored as tensors [3,32,32], labels as integer 0 or 1

In [3]:
label_map = {0: 0, 2: 1}
class_names = ['airplane', 'bird']
cifar2 = [(img, label_map[label]) for img, label in cifar10 if label in [0, 2]]
cifar2_test = [(img, label_map[label]) for img, label in cifar10_test if label in [0, 2]]

In [4]:
img, label=cifar2[1]
print(img.shape, label)
#cifar2

torch.Size([3, 32, 32]) 1


In [5]:
conv = nn.Conv2d(3, 16, kernel_size=3)
conv

Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))

In [6]:
conv.weight.shape, conv.bias.shape

(torch.Size([16, 3, 3, 3]), torch.Size([16]))

In [7]:
img, _ = cifar2[0]
output = conv(img.unsqueeze(0))
img.unsqueeze(0).shape, output.shape

(torch.Size([1, 3, 32, 32]), torch.Size([1, 16, 30, 30]))

In [8]:
pool = nn.MaxPool2d(2)
output = pool(img.unsqueeze(0))
img.unsqueeze(0).shape, output.shape

(torch.Size([1, 3, 32, 32]), torch.Size([1, 3, 16, 16]))

In [9]:

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        #self.act1 = nn.Tanh()
        #self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        #self.act2 = nn.Tanh()
        #self.pool2 = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(8 * 8 * 8, 32)
        #self.act3 = nn.Tanh()
        self.fc2 = nn.Linear(32, 2)
    def forward(self, x):
        out = self.conv1(x)
        out = torch.tanh(out) # function
        out = F.max_pool2d(out,2)
        out = self.conv2(out)
        out = torch.tanh(out)
        out = F.max_pool2d(out,2)
        out = out.view(-1, 8 * 8 * 8)
        out = self.fc1(out)
        out = torch.tanh(out)
        out = self.fc2(out)
        return out

In [10]:
model = Net()

In [11]:
numel_list = [p.numel() for p in model.parameters()]
sum(numel_list), numel_list

(18090, [432, 16, 1152, 8, 16384, 32, 64, 2])

In [12]:
print(img.unsqueeze(0).shape)
model(img.unsqueeze(0))

torch.Size([1, 3, 32, 32])


tensor([[ 0.0510, -0.0475]], grad_fn=<AddmmBackward>)

In [13]:
import datetime
def training_loop(n_epochs, optimizer, model, loss_fn, train_loader):
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            outputs = model(imgs)   # imgs: [batch_size, C,H,W] accepted by conv2d
            loss = loss_fn(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_train += loss.item()
        if epoch == 1 or epoch % 10 == 0:
            print('{} Epoch {}, Training loss {}'.format(
            datetime.datetime.now(), epoch,
            loss_train / len(train_loader)))

In [14]:
train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64,
    shuffle=True)
model = Net() #
optimizer = optim.SGD(model.parameters(), lr=1e-2) #
loss_fn = nn.CrossEntropyLoss()

In [15]:
training_loop(
n_epochs = 100,
optimizer = optimizer,
model = model,
loss_fn = loss_fn,
train_loader = train_loader,
)

2023-09-05 15:44:20.501572 Epoch 1, Training loss 0.643967665684451
2023-09-05 15:46:01.669627 Epoch 10, Training loss 0.37506938341316903
2023-09-05 15:47:47.958945 Epoch 20, Training loss 0.32019150741161057
2023-09-05 15:49:29.642442 Epoch 30, Training loss 0.2998704523037953
2023-09-05 15:52:55.265520 Epoch 40, Training loss 0.2842075728876576
2023-09-05 15:54:45.228890 Epoch 50, Training loss 0.2652401124026365
2023-09-05 15:56:25.186468 Epoch 60, Training loss 0.24640889541738353
2023-09-05 15:58:05.560932 Epoch 70, Training loss 0.23124178608132015
2023-09-05 15:59:46.228611 Epoch 80, Training loss 0.21609638830658737
2023-09-05 16:01:26.292905 Epoch 90, Training loss 0.20230945754962362
2023-09-05 16:03:07.255794 Epoch 100, Training loss 0.18751443300847034


In [16]:
len(train_loader)

157

In [17]:
test_loader = torch.utils.data.DataLoader(cifar2_test, batch_size=64,
shuffle=False)

In [18]:
def validate(model, train_loader, test_loader):
    for name, loader in [("train", train_loader), ("test", test_loader)]:
        correct = 0
        total = 0
        with torch.no_grad():
            for imgs, labels in loader:
                outputs = model(imgs)
                _, predicted = torch.max(outputs, dim=1)
                total += labels.shape[0]
                correct += int((predicted == labels).sum())
        print("Accuracy {}: {:.2f}".format(name , correct / total))

In [19]:
validate(model, train_loader, test_loader)

Accuracy train: 0.92
Accuracy test: 0.89


In [20]:
data_path = '../torch_tutorial/data'
torch.save(model.state_dict(), data_path + 'parameters.pt')

In [21]:
loaded_model = Net()
loaded_model.load_state_dict(torch.load(data_path
+ 'parameters.pt'))

<All keys matched successfully>

In [22]:
print(img.unsqueeze(0).shape)
loaded_model(img.unsqueeze(0))

torch.Size([1, 3, 32, 32])


tensor([[-0.8502,  1.5916]], grad_fn=<AddmmBackward>)